Supercharged mNeonGreen in mosaic — ProteinMPNN + Debye-Huckel    minimize   -log P_MPNN(s | fixed backbone)  +  lambda * E_DH(s)over the designable positions only. No structure prediction in the loop.**Why this runs as a script, not in cells.** mosaic pins `requires-python ="==3.12.*"` and declares eleven of its dependencies as git sources under`[tool.uv.sources]`, which plain pip does not read. Colab's kernel is on3.13, so `import mosaic` in a cell cannot work. uv supplies a 3.12interpreter and resolves the git sources; the design then runs as asubprocess under that interpreter.The whole lambda sweep happens inside one invocation, so the MPNN load andthe JAX compile are paid once rather than per lambda.

In [1]:
# ---- uv, a 3.12 interpreter, and mosaic's git dependencies ----
!pip install -q uv
!uv python install 3.12
!git clone -q https://github.com/escalante-bio/mosaic /content/mosaic
!cd /content/mosaic && uv sync --group jax-cuda --python 3.12

from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["MOSAIC_CACHE_DIR"] = "/content/drive/MyDrive/phbinder/mosaic_cache"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 222.3 MB/s eta 0:00:00
Installed Python 3.12.15 in 415ms
 + cpython-3.12.15-linux-x86_64-gnu (python3.12)
Using CPython 3.12.15
Creating virtual environment at: .venv
Resolved 244 packages in 3.66s
Prepared 235 packages in 30.12s
Installed 236 packages in 157ms
 + ablang==0.3.1
 + ablang2==0.2.1
 + absl-py==2.5.0
 + accelerate==1.15.0
 + aiofiles==25.1.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.4
 + aiosignal==1.4.0
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + anywidget==0.11.0
 + asttokens==3.0.2
 + attrs==26.1.0
 + awscrt==0.37.0
 + biopython==1.84
 + biotite==1.4.0
 + biotraj==1.2.2
 + boltz==2.2.1 (from git+https://github.com/escalante-bio/boltz.git@1acc397b6e81f30dc07a80d17a26c38f9c6f6942)
 + boltzgen==0.3.1 (from git+https://github.com/escalante-bio/boltzgen.git@c0bbc7ec0dad7f7bc11d28cd83297fb02ddb757a)
 + boto3==1.43.108
 + botocore==1.43.108
 + brotli==1.2.0
 + certifi==2026.7.22
 + ch

In [ ]:
!cd /content/mosaic && uv run --python 3.12 python -c "\
import mosaic, jax, gemmi; \
print('mosaic ok |', jax.__version__, jax.devices())"


Uninstalled 1 package in 0.33ms
Installed 1 package in 3ms
  File "<string>", line 1
    import mosaic, jax, gemmi;  # from mosaic.models.boltz2 import Boltz2;  # from mosaic.losses.protein_mpnn import InverseFoldingSequenceRecovery;  # from mosaic.losses.esmc import ESMCPseudoLikelihood, load_esmc;  # from mosaic.losses.transformations import ClippedLoss, NormedGradient;  # from mosaic.optimizers import simplex_APGM;  print('mosaic ok |', jax.__version__, jax.devices())
IndentationError: unexpected indent


In [3]:
%%writefile /content/ph_binder_design_boltz2.py
#!/usr/bin/env python3
"""pH-conditional EGFR binder: Boltz-2 + README objective. W_DH=0 disables the switch."""

import time
from pathlib import Path

import equinox as eqx
import gemmi
import jax
import jax.numpy as jnp
import numpy as np
from jaxtyping import Array, Float

import mosaic.losses.structure_prediction as sp
from mosaic.common import TOKENS, LossTerm
from mosaic.losses.esmc import ESMCPseudoLikelihood, load_esmc
from mosaic.losses.protein_mpnn import InverseFoldingSequenceRecovery
from mosaic.losses.stability import StabilityModel
from mosaic.losses.trigram import TrigramLL
from mosaic.models.boltz2 import Boltz2
from mosaic.optimizers import simplex_APGM, gradient_MCMC
from mosaic.proteinmpnn.mpnn import load_mpnn_sol
from mosaic.structure_prediction import TargetChain

# ======================================================================
# USER SETTINGS
# ======================================================================
BASE      = Path("/content/drive/MyDrive/adaptyv_comp1/egfr_ph")
PDB_PATH  = BASE / "TARGET.pdb"
Q65_PATH  = BASE / "q_target_65.npy"
Q74_PATH  = BASE / "q_target_74.npy"
OUT_FASTA = BASE / "ph_binder_designs_finalrun5.fa"
CHAIN     = "A"

TARGET_SEQUENCE = (
    "RKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLI"
    "QAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLF"
    "GTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCV"
)
RESNUM_OFFSET = 310

BINDER_LENGTH  = 70
N_DESIGNS      = 500
USE_TARGET_MSA = True
ESMC_MODEL     = "esmc_6b"

# --- interface, as (residue number, expected aa); asserted at startup ---
ANCHOR = [(323, "D"), (325, "L"), (344, "D"), (346, "H"), (370, "I"),
          (382, "L"), (384, "Q"), (406, "T"), (409, "H"), (415, "A"),
          (434, "D"), (436, "D"), (438, "I"), (465, "K"), (467, "I")]

CHARGE = [(322, "K"), (323, "D"), (344, "D"), (346, "H"), (409, "H"),
          (434, "D"), (436, "D"), (465, "K")]

ANCHOR_IDX = [r - RESNUM_OFFSET for r, _ in ANCHOR]
CHARGE_IDX = [r - RESNUM_OFFSET for r, _ in CHARGE]

# --- weights
W_CONTACT = 4.0
W_ROG     = 1.0
W_INTRA   = 1.0
W_HELIX   = 0.3
W_MPNN    = 5.0
W_ESMC    = 0.5
W_TRIGRAM = 1.0
W_STAB    = 0.1
W_MONOMER = 2
W_MAXHIS = 0.0

# metric is sqrt(2)*Rg
ROG_RADIUS = float(np.sqrt(2) * 2.38 * BINDER_LENGTH**0.365)

# --- pH switch (0.0 = off) ---
W_DH_BIND   = 0.1     # pulls E_on negative
W_DH_SWITCH = 25.0    # pulls E_on below E_off
PH_ON          = 6.5
PH_OFF         = 7.4
OFF_WEIGHT     = 1.0
IONIC_STRENGTH = 0.150
EPS_R          = 78.5
TEMPERATURE    = 298.15

# --- optimizer (README values) ---
N_STEPS_SOFT = 75
N_STEPS_MID  = 75
N_STEPS_SHARP = 50
SCALE_SOFT   = 1.0
SCALE_MID    = 1.1
SCALE_SHARP  = 1.6
MCMC_STEPS = 50
MCMC_TEMP  = 0.001     # gradient_MCMC default
STEPSIZE        = 0.1
MOMENTUM        = 0.9
RECYCLING_STEPS = 1
# ======================================================================

_E, _KB, _EPS0, _NA = 1.602176634e-19, 1.380649e-23, 8.8541878128e-12, 6.02214076e23

BINDER_PKA = {"K": (10.5, +1), "R": (12.5, +1), "H": (6.0, +1),
              "D": (3.8, -1), "E": (4.5, -1), "C": (9.0, -1), "Y": (10.0, -1)}


def debye_kappa(I, eps_r=EPS_R, T=TEMPERATURE):
    """Inverse Debye length, 1/Angstrom."""
    return np.sqrt(2.0 * I * 1e3 * _NA * _E**2 / (eps_r * _EPS0 * _KB * T)) * 1e-10


def binder_charge_vector(pH):
    """HH charge per token, TOKENS order."""
    q = np.zeros(len(TOKENS))
    for aa, (pKa, sign) in BINDER_PKA.items():
        k = TOKENS.index(aa)
        if sign > 0:
            q[k] = 1.0 / (1.0 + 10 ** (pH - pKa))
        else:
            q[k] = -1.0 / (1.0 + 10 ** (pKa - pH))
    return q

class MaxHistidine(LossTerm):
    max_count: float = 4.0

    def __call__(self, sequence: Float[Array, "N 20"], key=None):
        n = sequence[:, TOKENS.index("H")].sum()
        v = jax.nn.relu(n - self.max_count)
        return v, {"n_his": n}

class DebyeHuckelInterface(LossTerm):
    """Screened Coulomb energy, binder to masked target residues.

    Per-pair energy is gated by contact probability so an uncertain distogram
    cannot harvest most of a real contact's reward. Reduction is per TARGET
    residue -- each charge target keeps its strongest per_target_k binder
    partners -- so a crowded region of the epitope cannot monopolise the term
    and leave isolated target charges with no gradient. Sorting by magnitude
    keeps repulsive pairs in the sum.
    """

    q_target: Float[Array, " M"] = eqx.field(converter=jnp.array)
    q_tokens: Float[Array, " 20"] = eqx.field(converter=jnp.array)
    epitope_idx: tuple
    kappa: float
    prefactor: float
    cutoff: float = 12.0
    min_dist: float = 4.0
    gate_power: float = 5.0
    per_target_k: int = 4
    name: str = "E_dh"

    def __call__(self, sequence: Float[Array, "N 20"], output, key):
        binder_len = sequence.shape[0]
        logits = output.distogram_logits[:binder_len, binder_len:]
        logits = logits[:, jnp.asarray(self.epitope_idx)]        # [Lb, M, B]
        p = jax.nn.softmax(logits, axis=-1)

        r = output.distogram_bins
        near = (r < self.cutoff).astype(p.dtype)
        kernel = near * jnp.exp(-self.kappa * r) / jnp.maximum(r, self.min_dist)

        e_pair = (p * kernel).sum(-1)                            # [Lb, M]
        p_contact = (p * near).sum(-1)                           # [Lb, M]

        q_b = sequence @ self.q_tokens                           # [Lb]
        contrib = (q_b[:, None] * self.q_target[None, :]
                   * e_pair * p_contact ** self.gate_power)      # [Lb, M]

        # per target residue, keep its strongest per_target_k binder partners
        order = jnp.argsort(-jnp.abs(contrib), axis=0)           # [Lb, M]
        keep = jnp.take_along_axis(contrib, order[: self.per_target_k], axis=0)
        E = self.prefactor * keep.sum()
        return E, {self.name: E}


def main():

    q65_full = np.load(Q65_PATH)
    q74_full = np.load(Q74_PATH)
    assert len(q65_full) == len(TARGET_SEQUENCE), (
        f"charge vector is {len(q65_full)} long, TARGET_SEQUENCE is "
        f"{len(TARGET_SEQUENCE)}; same numbering required")

    for label, table in (("ANCHOR", ANCHOR), ("CHARGE", CHARGE)):
        for resnum, expected in table:
            got = TARGET_SEQUENCE[resnum - RESNUM_OFFSET]
            assert got == expected, (
                f"{label}: residue {resnum} is {got}, not {expected}")

    st = gemmi.read_structure(str(PDB_PATH))
    st.setup_entities()
    template = st[0][CHAIN]
    tmpl_seq = gemmi.one_letter_code([r.name for r in template]).upper()
    assert tmpl_seq == TARGET_SEQUENCE, (
        "template chain and TARGET_SEQUENCE disagree:\n"
        f"  template {len(tmpl_seq)}: {tmpl_seq}\n"
        f"  expected {len(TARGET_SEQUENCE)}: {TARGET_SEQUENCE}")

    kappa = debye_kappa(IONIC_STRENGTH)
    prefactor = 560.0 / EPS_R

    q65 = q65_full[CHARGE_IDX]
    q74 = q74_full[CHARGE_IDX]

    print(f"binder {BINDER_LENGTH} aa vs target {len(TARGET_SEQUENCE)} aa, "
          f"templated on {PDB_PATH.name}")
    print(f"anchor {len(ANCHOR_IDX)} residues: "
          f"{' '.join(a + str(r) for r, a in ANCHOR)}")
    print(f"Rg target {ROG_RADIUS:.2f} (metric = sqrt(2) * "
          f"{ROG_RADIUS/np.sqrt(2):.2f} A true Rg)")
    if W_DH_BIND or W_DH_SWITCH:
        print(f"W_DH_BIND={W_DH_BIND}  W_DH_SWITCH={W_DH_SWITCH}  "
              f"kappa {kappa:.4f} 1/A (Debye {1/kappa:.2f} A)")
        for (resnum, aa), a, b in zip(CHARGE, q65, q74):
            print(f"  {aa}{resnum:<4} q({PH_ON})={a:+.3f}  q({PH_OFF})={b:+.3f}  "
                  f"dq={a-b:+.3f}")
        print(f"  total dq {float((q65 - q74).sum()):+.3f}")
    else:
        print("pH switch OFF (W_DH = 0) -- standard binder run")
    print(flush=True)

    boltz2 = Boltz2()
    mpnn = load_mpnn_sol(0.05)
    esmc = load_esmc(ESMC_MODEL)
    esmc_stab = load_esmc("esmc_300m")
    trigram_ll = TrigramLL.from_pkl()
    stability_loss = StabilityModel.from_pretrained(esmc_stab)
    print(f"Boltz-2 + soluble MPNN + {ESMC_MODEL} + trigram + stability loaded",
          flush=True)

    target_chains = [TargetChain(sequence=TARGET_SEQUENCE,
                                 use_msa=USE_TARGET_MSA,
                                 template_chain=template)]

    features, _ = boltz2.binder_features(BINDER_LENGTH, target_chains)
    mono_features, _ = boltz2.binder_features(BINDER_LENGTH, [])
    print("features built", flush=True)

    dh_on = DebyeHuckelInterface(
        q_target=q65, q_tokens=binder_charge_vector(PH_ON),
        epitope_idx=tuple(CHARGE_IDX), kappa=kappa,
        prefactor=prefactor, name="E_on")
    dh_off = DebyeHuckelInterface(
        q_target=q74, q_tokens=binder_charge_vector(PH_OFF),
        epitope_idx=tuple(CHARGE_IDX), kappa=kappa,
        prefactor=prefactor, name="E_off")

    complex_loss = (
        W_CONTACT * sp.BinderTargetContact(epitope_idx=ANCHOR_IDX)
        + W_ROG * sp.DistogramRadiusOfGyration(target_radius=ROG_RADIUS)
        + W_INTRA * sp.WithinBinderContact()
        + W_HELIX * sp.HelixLoss()
        + W_MPNN * InverseFoldingSequenceRecovery(mpnn, temp=jnp.array(0.01))
    )
    if W_DH_BIND or W_DH_SWITCH:
        complex_loss = (complex_loss
                        + W_DH_BIND * dh_on
                        + W_DH_SWITCH * (dh_on - dh_off))

    # binder alone
    monomer_loss = (
        0.2 * sp.PLDDTLoss()
        + sp.DistogramRadiusOfGyration(target_radius=ROG_RADIUS)
        + W_HELIX * sp.HelixLoss()
    )

    loss = (
        boltz2.build_loss(loss=complex_loss, features=features,
                          recycling_steps=RECYCLING_STEPS)
        + W_ESMC * ESMCPseudoLikelihood(esmc)
        + W_TRIGRAM * trigram_ll
        + W_STAB * stability_loss
        + W_MONOMER * boltz2.build_loss(loss=monomer_loss,
                                        features=mono_features,
                                        recycling_steps=RECYCLING_STEPS)
        + W_MAXHIS * MaxHistidine(max_count=4.0)
    )

    OUT_FASTA.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()


    for j in range(N_DESIGNS):
        seed = int(np.random.randint(100000))
        x = jax.nn.softmax(0.5 * jax.random.gumbel(
            key=jax.random.key(seed), shape=(BINDER_LENGTH, len(TOKENS))))

        print(f"\n=== design {j + 1}/{N_DESIGNS}  seed {seed}  "
              f"stepsize {STEPSIZE}  momentum {MOMENTUM} ===", flush=True)
        _, x = simplex_APGM(loss_function=loss, x=x, n_steps=N_STEPS_SOFT,
                            stepsize=STEPSIZE, momentum=MOMENTUM,
                            scale=SCALE_SOFT)
        _, pssm = simplex_APGM(loss_function=loss, x=x, n_steps=N_STEPS_MID,
                            stepsize=STEPSIZE, momentum=MOMENTUM,
                            scale=SCALE_MID)
        # _, pssm = simplex_APGM(loss_function=loss, x=x, n_steps=N_STEPS_SHARP,
        #                        stepsize=STEPSIZE, momentum=MOMENTUM,
        #                        scale=SCALE_SHARP)
        seq_idx = jnp.argmax(pssm, axis=-1).astype(jnp.int32)
        if MCMC_STEPS:
            seq_idx = gradient_MCMC(loss, seq_idx, steps=MCMC_STEPS,
                                    temp=MCMC_TEMP,
                                    key=jax.random.key(seed + 1))
        seq = "".join(TOKENS[int(i)] for i in np.asarray(seq_idx))
        netq = sum(a in "KR" for a in seq) - sum(a in "DE" for a in seq)
        nhis = seq.count("H")
        onehot = jax.nn.one_hot(seq_idx, 20)
        _, final_aux = loss(onehot, key=jax.random.key(0))
        plddt = [float(v) for p, v in jax.tree_util.tree_flatten_with_path(final_aux)[0]
                 if "plddt" in str(p)][0]
        tag = f"b{W_DH_BIND}s{W_DH_SWITCH}"
        with open(OUT_FASTA, "a") as fh:
            fh.write(f">phbinder|{tag}|seed={seed}|plddt={plddt:.3f}"
                     f"|netQ={netq:+d}|nH={nhis}\n{seq}\n")
        print(f"netQ {netq:+3d}  nH {nhis:2d}  {seq}", flush=True)

    print(f"\n{N_DESIGNS} designs in {time.time() - t0:.0f}s -> {OUT_FASTA}")


if __name__ == "__main__":
    main()

Overwriting /content/ph_binder_design_boltz2.py


In [ ]:
!cd /content/mosaic && MOSAIC_CACHE_DIR=$MOSAIC_CACHE_DIR \
    uv run --python 3.12 python /content/ph_binder_design_boltz2.py

Streaming output truncated to the last 5000 lines.
50 loss: -11.45 .0.boltz2.0.target_contact:-2.24 .0.boltz2.1.radius_of_gyration: 15.32 .0.boltz2.2.intra_contact:-3.28 .0.boltz2.3.helix:-0.08 .0.boltz2.4.sequence_recovery: 0.28 .0.boltz2.5.E_on:-1.07 .0.boltz2.6.E_on:-1.07 .0.boltz2.7.E_off:-0.09 .1.esmc_pll:-2.83 .2.trigram_ll:-3.19 .3.delta_g: 3.00 .4.boltz2.0.plddt: 0.82 .4.boltz2.1.radius_of_gyration: 15.22 .4.boltz2.2.helix:-0.36 .5.n_his: 5.19 loss:-11.45 nnz: 5.11 time: 2.25
51 loss: -11.40 .0.boltz2.0.target_contact:-2.35 .0.boltz2.1.radius_of_gyration: 15.25 .0.boltz2.2.intra_contact:-3.20 .0.boltz2.3.helix:-0.19 .0.boltz2.4.sequence_recovery: 0.29 .0.boltz2.5.E_on:-1.19 .0.boltz2.6.E_on:-1.19 .0.boltz2.7.E_off:-0.20 .1.esmc_pll:-2.85 .2.trigram_ll:-3.18 .3.delta_g: 2.08 .4.boltz2.0.plddt: 0.83 .4.boltz2.1.radius_of_gyration: 15.21 .4.boltz2.2.helix:-0.41 .5.n_his: 5.09 loss:-11.40 nnz: 5.19 time: 2.25
52 loss: -10.59 .0.boltz2.0.target_contact:-2.29 .0.boltz2.1.radius_of_gy

In [1]:
%%writefile /content/repredict.py
#!/usr/bin/env python3
"""Repredict each design against the human AND mouse target, reporting iPTM, pLDDT,
PAE and Debye-Huckel energies for both."""

from pathlib import Path

import gemmi
import jax
import jax.numpy as jnp
import numpy as np

from mosaic.common import TOKENS
from mosaic.models.boltz2 import Boltz2
from mosaic.structure_prediction import TargetChain
from mosaic.losses.structure_prediction import IPTMLoss

# ======================================================================
BASE       = Path("/content/drive/MyDrive/adaptyv_comp1/egfr_ph")
IN_FASTA   = BASE / "ph_binder_designs_ALL.fa"
OUT_DIR    = BASE / "REPREDICTIONS_ALLRUNS_FINAL"
CHAIN      = "A"

HUMAN_PDB  = BASE / "TARGET.pdb"
HUMAN_Q65  = BASE / "q_target_65.npy"
HUMAN_Q74  = BASE / "q_target_74.npy"

MOUSE_PDB  = BASE / "mouseTARGET.pdb"
MOUSE_Q65  = BASE / "q_mouse_65.npy"
MOUSE_Q74  = BASE / "q_mouse_74.npy"

TOP_N = 1000                # how many of the best-pLDDT designs to repredict

HUMAN_SEQUENCE = (
    "RKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLI"
    "QAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLF"
    "GTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCV"
)
MOUSE_SEQUENCE = (
    "RKVCNGIGIGEFKDTLSINATNIKHFKYCTAISGDLHILPVAFKGDSFTRTPPLDPRELEILKTVKEITGFLLI"
    "QAWPDNWTDLHAFENLEIIRGRTKQHGQFSLAVVGLNITSLGLRSLKEISDGDVIISGNRNLCYANTINWKKLF"
    "GTPNQKTKIMNNRAEKDCKAVNHVCNPLCSSEGCWGPEPRDCV"
)

# gapless 1:1 alignment, so one index set serves both species
ANCHOR_IDX = [13, 15, 34, 36, 60, 72, 74, 96, 99, 105, 124, 126, 128, 153, 155, 157]
CHARGE_IDX = [12, 13, 34, 36, 99, 124, 126, 153, 155]

USE_TARGET_MSA  = True
RECYCLING_STEPS = 20
SAMPLING_STEPS  = 100
SEED            = 0

PH_ON, PH_OFF  = 6.5, 7.4
IONIC_STRENGTH = 0.150
EPS_R          = 78.5
TEMPERATURE    = 298.15
# ======================================================================

_E, _KB, _EPS0, _NA = 1.602176634e-19, 1.380649e-23, 8.8541878128e-12, 6.02214076e23
BINDER_PKA = {"K": (10.5, +1), "R": (12.5, +1), "H": (6.0, +1),
              "D": (3.8, -1), "E": (4.5, -1), "C": (9.0, -1), "Y": (10.0, -1)}


def debye_kappa(I, eps_r=EPS_R, T=TEMPERATURE):
    return np.sqrt(2.0 * I * 1e3 * _NA * _E**2 / (eps_r * _EPS0 * _KB * T)) * 1e-10


def binder_residue_charges(seq, pH):
    q = np.zeros(len(seq))
    for i, a in enumerate(seq):
        if a in BINDER_PKA:
            pKa, sign = BINDER_PKA[a]
            q[i] = (1.0 / (1.0 + 10 ** (pH - pKa)) if sign > 0
                    else -1.0 / (1.0 + 10 ** (pKa - pH)))
    return q


def header_field(name, key, default=float("nan")):
    for tok in name.split("|"):
        if tok.startswith(key + "="):
            try:
                return float(tok.split("=", 1)[1])
            except ValueError:
                return default
    return default


def read_fasta(path):
    out, name, seq = [], None, []
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if name is not None:
                out.append((name, "".join(seq)))
            name, seq = line[1:], []
        else:
            seq.append(line)
    if name is not None:
        out.append((name, "".join(seq)))
    return out


def sc_coords(chain):
    """CB where present (sidechains exist with target_only_features), else CA."""
    xyz = []
    for r in chain:
        a = r.find_atom("CB", "*")
        if a is None:
            a = r.find_atom("CA", "*")
        if a is not None:
            xyz.append([a.pos.x, a.pos.y, a.pos.z])
    return np.array(xyz)


def dh_energy(bx, tx, qb, qt, kappa, prefactor):
    d = np.linalg.norm(bx[:, None, :] - tx[None, :, :], axis=-1)
    d = np.maximum(d, 4.0)
    return float(prefactor * (qb[:, None] * qt[None, :] * np.exp(-kappa * d) / d).sum())


def load_target(label, pdb_path, sequence, q65_path, q74_path):
    st = gemmi.read_structure(str(pdb_path))
    st.setup_entities()
    template = st[0][CHAIN]
    tmpl_seq = gemmi.one_letter_code([r.name for r in template]).upper()
    assert tmpl_seq == sequence, (
        f"{label}: template chain and sequence disagree\n"
        f"  template {len(tmpl_seq)}: {tmpl_seq}\n"
        f"  expected {len(sequence)}: {sequence}")

    if q65_path.exists() and q74_path.exists():
        q65, q74 = np.load(q65_path), np.load(q74_path)
        assert len(q65) == len(sequence), (
            f"{label}: charge vector {len(q65)} vs sequence {len(sequence)}")
        print(f"{label}: template + charge vectors loaded")
    else:
        q65 = q74 = None
        print(f"{label}: no charge vectors ({q65_path.name}), energies will be NaN")

    return {"label": label, "template": template, "sequence": sequence,
            "q65": q65, "q74": q74}


def score_against(boltz2, target, seq, kappa, prefactor):
    binder_length = len(seq)
    chains = [
        TargetChain(sequence=seq, use_msa=False),
        TargetChain(sequence=target["sequence"], use_msa=USE_TARGET_MSA,
                    template_chain=target["template"]),
    ]
    features, writer = boltz2.target_only_features(chains)
    full = seq + target["sequence"]
    pssm = jax.nn.one_hot(jnp.array([TOKENS.index(a) for a in full]), 20)
    pred = boltz2.predict(PSSM=pssm, features=features, writer=writer,
                          key=jax.random.key(SEED),
                          recycling_steps=RECYCLING_STEPS,
                          sampling_steps=SAMPLING_STEPS)

    plddt = np.asarray(pred.plddt)
    pae = np.asarray(pred.pae)
    bt = pae[:binder_length, binder_length:]
    out = {
        "iptm": float(-IPTMLoss()(pssm[:binder_length], pred.model_output,
                                  jax.random.key(0))[0]),
        "binder_plddt": float(plddt[:binder_length].mean()),
        "anchor_pae": float(bt[:, ANCHOR_IDX].mean()),
        "charge_pae": float(bt[:, CHARGE_IDX].mean()),
    }

    model = pred.st[0]
    bx, tx = sc_coords(model[0]), sc_coords(model[1])
    if target["q65"] is not None and len(tx) == len(target["q65"]):
        e_on = dh_energy(bx, tx, binder_residue_charges(seq, PH_ON),
                         target["q65"], kappa, prefactor)
        e_off = dh_energy(bx, tx, binder_residue_charges(seq, PH_OFF),
                          target["q74"], kappa, prefactor)
        out.update(E_on=e_on, E_off=e_off, dE=e_on - e_off)
    else:
        out.update(E_on=float("nan"), E_off=float("nan"), dE=float("nan"))

    return out, pred.st


def main():
    designs = read_fasta(IN_FASTA)
    if not designs:
        raise SystemExit(f"no sequences in {IN_FASTA}")

    # rank by the pLDDT recorded in the header; designs without one go last
    ranked = sorted(designs,
                    key=lambda nv: (-(header_field(nv[0], "plddt")
                                      if header_field(nv[0], "plddt") == header_field(nv[0], "plddt")
                                      else -1.0)))
    n_with = sum(1 for n, _ in designs
                 if header_field(n, "plddt") == header_field(n, "plddt"))
    print(f"{len(designs)} designs in file, {n_with} with pLDDT in header")

    ranked = ranked[:TOP_N]
    print(f"repredicting top {len(ranked)} against 2 targets "
          f"({2 * len(ranked)} predictions)", flush=True)

    kappa = debye_kappa(IONIC_STRENGTH)
    prefactor = 560.0 / EPS_R

    targets = [
        load_target("human", HUMAN_PDB, HUMAN_SEQUENCE, HUMAN_Q65, HUMAN_Q74),
        load_target("mouse", MOUSE_PDB, MOUSE_SEQUENCE, MOUSE_Q65, MOUSE_Q74),
    ]

    boltz2 = Boltz2()
    print("Boltz-2 loaded", flush=True)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    csv = OUT_DIR / "repredict.csv"
    cols = (["rank", "name", "design_plddt", "netQ", "nH"]
            + [f"{t['label'][0]}_{k}" for t in targets
               for k in ("iptm", "plddt", "anchor_pae", "charge_pae",
                         "E_on", "E_off", "dE")]
            + ["d_iptm", "seq"])
    with open(csv, "w") as fh:
        fh.write(",".join(cols) + "\n")

    for j, (name, seq) in enumerate(ranked):
        row = {"rank": j, "name": name,
               "design_plddt": header_field(name, "plddt"),
               "netQ": sum(a in "KR" for a in seq) - sum(a in "DE" for a in seq),
               "nH": seq.count("H"), "seq": seq}

        for t in targets:
            res, st_pred = score_against(boltz2, t, seq, kappa, prefactor)
            p = t["label"][0]
            row[f"{p}_iptm"] = res["iptm"]
            row[f"{p}_plddt"] = res["binder_plddt"]
            row[f"{p}_anchor_pae"] = res["anchor_pae"]
            row[f"{p}_charge_pae"] = res["charge_pae"]
            row[f"{p}_E_on"] = res["E_on"]
            row[f"{p}_E_off"] = res["E_off"]
            row[f"{p}_dE"] = res["dE"]
            st_pred.write_pdb(str(OUT_DIR / f"rank{j:03d}_{t['label']}.pdb"))

        row["d_iptm"] = row["h_iptm"] - row["m_iptm"]

        with open(csv, "a") as fh:
            fh.write(",".join(
                f"{row[c]:.4f}" if isinstance(row[c], float) else str(row[c])
                for c in cols) + "\n")

        print(f"[{j+1}/{len(ranked)}] L={len(seq)}  "
              f"human iptm {row['h_iptm']:.3f} plddt {row['h_plddt']:.3f} "
              f"dE {row['h_dE']:+6.2f}  |  "
              f"mouse iptm {row['m_iptm']:.3f} plddt {row['m_plddt']:.3f} "
              f"dE {row['m_dE']:+6.2f}  |  d_iptm {row['d_iptm']:+.3f}",
              flush=True)

    print(f"\nwrote {csv}")


if __name__ == "__main__":
    main()

Writing /content/repredict.py


In [ ]:
!cd /content/mosaic && MOSAIC_CACHE_DIR=$MOSAIC_CACHE_DIR \
    uv run --python 3.12 python /content/repredict.py

Streaming output truncated to the last 5000 lines.
Something odd happened with manifest, trying to reload.
/content/mosaic/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Loading data
Checking input data.
Processing 1 inputs with 1 threads.
  0% 0/1 [00:00<?, ?it/s]Found explicit empty MSA for some proteins, will run these in single sequence mode. Keep in mind that the model predictions will be suboptimal without an MSA.
Generating MSA for /tmp/tmpw7jsp5uy/input.yaml with 1 protein entities.
Calling MSA server for target input with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server

  0% 0/150 [00:00<?, ?it/s]
SUBMIT:   0% 0/150 [00:00<?, ?it/s]
COMPLETE:   0% 0/150 [00:00<?, ?it/s]
COMPLETE: 100% 150/150 [00:01<00:00, 85.30it/s] 
100% 1/1 [00:01

In [ ]:
# from google.colab import files
# files.download('/content/drive/MyDrive/adaptyv_comp1/egfr_ph/repredict/design_00.pdb')
# files.download('/content/drive/MyDrive/adaptyv_comp1/egfr_ph/repredict/repredict.csv')
!cd /content/mosaic && uv run --python 3.12 python -c "import torch; torch.load('/content/drive/MyDrive/phbinder/mosaic_cache/boltz/boltz2_conf.ckpt', map_location='cpu', weights_only=False); print('ckpt ok')"

ckpt ok
